# 06 — Integración ENOE × INPC → Panel Analítico (Plata)

**Propósito:** Cruzar ENOE Plata con INPC para calcular salarios reales
(deflactados, base diciembre 2018) y producir el conjunto analítico
listo para consumo en la capa Plata.

**Pipeline:** `ingesta_general.ipynb` → `05_limpieza_enoe.ipynb` → **`06_integracion.ipynb`**

**Prerequisitos:**
- `ingesta_general.ipynb` ejecutado (bronce ENOE, INPC, SHCP, IGAE)
- `05_limpieza_enoe.ipynb` ejecutado (plata ENOE limpia)

**Granularidad de salida:** persona × trimestre (misma que ENOE)
**Ubicación:** `data/silver/panel_enoe_inpc/`

In [ ]:
# ── Importaciones ───────────────────────────────────────────────────────────
import os
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

spark = (
    SparkSession.builder
    .appName('06_integracion')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark', spark.version)

In [ ]:
# ── Rutas ───────────────────────────────────────────────────────────────────
BASE   = Path(os.getcwd()).parent
RAW    = BASE / 'data' / 'raw'
SILVER = BASE / 'data' / 'silver'

ENOE_SILVER  = str(SILVER / 'enoe')               # salida de 05_limpieza_enoe
INPC_PARQUET = str(RAW    / 'inpc' / 'parquet')   # bronce INPC
IGAE_PARQUET = str(RAW    / 'igae' / 'parquet')   # bronce IGAE
PANEL_OUT    = str(SILVER / 'panel_enoe_inpc')    # salida de este notebook

Path(PANEL_OUT).mkdir(parents=True, exist_ok=True)

print('ENOE Plata  :', ENOE_SILVER)
print('INPC Bronce :', INPC_PARQUET)
print('IGAE Bronce :', IGAE_PARQUET)
print('Panel salida:', PANEL_OUT)

---
## 1 · Extracción — leer fuentes

Se cargan ENOE (capa Plata) e INPC (capa Bronce).
Se registran conteos y granularidad de entrada.

In [ ]:
# ── 1.1 Cargar ENOE Plata ───────────────────────────────────────────────────
df_enoe = spark.read.parquet(ENOE_SILVER)

filas_enoe = df_enoe.count()
print(f'ENOE Plata  — filas    : {filas_enoe:,}')
print(f'ENOE Plata  — columnas : {len(df_enoe.columns)}')
print(f'ENOE Plata  — periodo  :',
      df_enoe.agg(F.min('anio'), F.max('anio')).collect()[0][:])

In [ ]:
# ── 1.2 Cargar INPC (subíndice general) ────────────────────────────────────
# Solo necesitamos el INPC general para deflactar.
# El subíndice que representa el nivel general es
# 'Precios al Consumidor (INPC)' en la columna subindice.
df_inpc_raw = spark.read.parquet(INPC_PARQUET)

print('Subíndices disponibles en INPC:')
df_inpc_raw.select('subindice').distinct().orderBy('subindice').show(truncate=False)

In [ ]:
# ── 1.3 Filtrar INPC general y calcular promedio trimestral ─────────────────
# La ENOE tiene granularidad trimestral; el INPC es mensual.
# Se promedia el INPC general por trimestre para hacer el join.

SUBINDICE_GENERAL = 'Precios al Consumidor (INPC)'

df_inpc = (
    df_inpc_raw
    .filter(F.col('subindice') == SUBINDICE_GENERAL)
    .groupBy('anio', 'trimestre')
    .agg(F.avg('indice_valor').alias('inpc_trim'))
    .orderBy('anio', 'trimestre')
)

print(f'Trimestres INPC disponibles: {df_inpc.count()}')
df_inpc.show(20, truncate=False)

In [ ]:
# ── 1.4 Cargar IGAE (actividad económica agregada) ──────────────────────────
# Se usará el IGAE Total promediado por trimestre para enriquecer el panel.
df_igae_raw = spark.read.parquet(IGAE_PARQUET)

df_igae = (
    df_igae_raw
    .filter(F.col('ACTIVIDAD') == 'Total')
    .withColumnRenamed('CICLO', 'anio')
    .withColumnRenamed('TRIMESTRE', 'trimestre')
    .groupBy('anio', 'trimestre')
    .agg(F.avg('IGAE').alias('igae_total'))
    .orderBy('anio', 'trimestre')
)

print(f'Trimestres IGAE disponibles: {df_igae.count()}')
df_igae.show(20, truncate=False)

---
## 2 · Transformación — deflactación y enriquecimiento

Se calculan:
- **`inpc_base`**: valor del INPC en el trimestre de referencia (T4-2018)
- **`salario_real`**: `ingocup × (inpc_base / inpc_trim)` — salario real base dic 2018
- **`ing7c_real`**: deflactación del ingreso de trabajo (incluye autoconsumo)

> **Decisión:** Se usa dic-2018 como base porque el IGAE usa base 2018=100,
> permitiendo comparar magnitudes entre fuentes.

In [ ]:
# ── 2.1 Obtener INPC base (promedio T4-2018) ────────────────────────────────
inpc_base_row = (
    df_inpc
    .filter((F.col('anio') == 2018) & (F.col('trimestre') == 4))
    .select('inpc_trim')
    .collect()
)

if inpc_base_row:
    INPC_BASE = inpc_base_row[0]['inpc_trim']
else:
    # Fallback: usar el primer trimestre disponible como base
    INPC_BASE = df_inpc.agg(F.min('inpc_trim')).collect()[0][0]
    print('ADVERTENCIA: T4-2018 no encontrado, usando mínimo como base')

print(f'INPC base (T4-2018): {INPC_BASE:.4f}')

In [ ]:
# ── 2.2 Unir ENOE con INPC trimestral ──────────────────────────────────────
df_panel = df_enoe.join(
    df_inpc.select('anio', 'trimestre', 'inpc_trim'),
    on=['anio', 'trimestre'],
    how='left'
)

# Verificar que el join no generó filas extras (no debe haber fan-out)
filas_post_join = df_panel.count()
print(f'Filas pre-join  : {filas_enoe:,}')
print(f'Filas post-join : {filas_post_join:,}')
if filas_post_join != filas_enoe:
    print('ADVERTENCIA: el join cambió el conteo de filas')
else:
    print('OK — sin fan-out')

# Nulos en inpc_trim (trimestres ENOE sin INPC correspondiente)
nulos_inpc = df_panel.filter(F.col('inpc_trim').isNull()).count()
print(f'Trimestres sin INPC: {nulos_inpc:,} filas')

In [ ]:
# ── 2.3 Calcular salario real deflactado ────────────────────────────────────
df_panel = df_panel.withColumn(
    'salario_real',
    F.when(
        F.col('ingocup').isNotNull() & F.col('inpc_trim').isNotNull() & (F.col('inpc_trim') > 0),
        F.round(F.col('ingocup') * (F.lit(INPC_BASE) / F.col('inpc_trim')), 2)
    ).otherwise(F.lit(None).cast(DoubleType()))
).withColumn(
    'ing7c_real',
    F.when(
        F.col('ing7c').isNotNull() & F.col('inpc_trim').isNotNull() & (F.col('inpc_trim') > 0),
        F.round(F.col('ing7c') * (F.lit(INPC_BASE) / F.col('inpc_trim')), 2)
    ).otherwise(F.lit(None).cast(DoubleType()))
)

print('Columnas de salario real creadas OK')
df_panel.select('anio', 'trimestre', 'ingocup', 'inpc_trim', 'salario_real').show(10)

In [ ]:
# ── 2.4 Unir IGAE (contexto macroeconómico) ────────────────────────────────
df_panel = df_panel.join(
    df_igae.select('anio', 'trimestre', 'igae_total'),
    on=['anio', 'trimestre'],
    how='left'
)

print(f'Columnas finales del panel: {len(df_panel.columns)}')
print(df_panel.columns)

---
## 3 · Validación del conjunto integrado

Se verifican la granularidad, integridad del join y consistencia
de las variables deflactadas antes de guardar.

In [ ]:
# ── 3.1 Granularidad: una fila por persona × trimestre ──────────────────────
JOIN_KEYS = [
    'r_def', 'cd_a', 'ent', 'con', 'upm', 'd_sem',
    'n_pro_viv', 'v_sel', 'n_hog', 'h_mud', 'n_ent', 'per', 'n_ren'
]

total      = df_panel.count()
distintos  = df_panel.dropDuplicates(JOIN_KEYS + ['anio', 'trimestre']).count()
print(f'Total filas       : {total:,}')
print(f'Filas únicas      : {distintos:,}')
print(f'Duplicados        : {total - distintos:,}  (esperado: 0)')

In [ ]:
# ── 3.2 Verificar salario_real vs ingocup nominal ───────────────────────────
# Para ocupados (clase1=1) con ingreso > 0, el salario real debe ser positivo.
(
    df_panel
    .filter((F.col('clase1') == 1) & (F.col('ingocup') > 0))
    .select(
        F.count('*').alias('ocupados_con_ingreso'),
        F.round(F.avg('ingocup'),     2).alias('ingocup_nominal_prom'),
        F.round(F.avg('salario_real'),2).alias('salario_real_prom'),
        F.count(F.when(F.col('salario_real').isNull(), True)).alias('sin_deflactar')
    )
    .show(truncate=False)
)

In [ ]:
# ── 3.3 Distribución por clase de actividad ─────────────────────────────────
# clase1: 1=ocupados, 2=desocupados, 3=no económicamente activos
total_panel = df_panel.count()
(
    df_panel
    .groupBy('clase1')
    .agg(
        F.count('*').alias('filas'),
        F.round(F.count('*') * 100.0 / total_panel, 2).alias('pct')
    )
    .orderBy('clase1')
    .show()
)

In [ ]:
# ── 3.4 Tendencia de salario real promedio por año (ocupados) ───────────────
print('Salario real promedio anual (ocupados con ingreso > 0):')
(
    df_panel
    .filter((F.col('clase1') == 1) & (F.col('salario_real') > 0))
    .groupBy('anio')
    .agg(
        F.round(F.avg('salario_real'), 2).alias('salario_real_prom'),
        F.round(F.avg('igae_total'),   2).alias('igae_prom'),
        F.count('*').alias('ocupados')
    )
    .orderBy('anio')
    .show(30, truncate=False)
)

---
## 4 · Almacenamiento — Panel Analítico Plata

Se guarda el panel integrado como Parquet en `data/silver/panel_enoe_inpc/`,
particionado por `anio`. Este es el conjunto de datos listo para análisis.

In [ ]:
# ── 4.1 Escribir Panel Analítico Plata ──────────────────────────────────────
(
    df_panel
    .write
    .mode('overwrite')
    .partitionBy('anio')
    .parquet(PANEL_OUT)
)

size_mb = sum(
    f.stat().st_size for f in Path(PANEL_OUT).rglob('*.parquet')
) / 1_048_576
print(f'Panel Plata escrito en : {PANEL_OUT}')
print(f'Tamaño estimado        : {size_mb:.1f} MB (snappy)')

In [ ]:
# ── 4.2 Validación round-trip ───────────────────────────────────────────────
df_final = spark.read.parquet(PANEL_OUT)

print('Validación round-trip:')
print(f'  Filas releídas : {df_final.count():,}')
print(f'  Columnas       : {len(df_final.columns)}')
df_final.printSchema()

In [ ]:
# ── 4.3 Resumen final del pipeline ──────────────────────────────────────────
filas_panel = df_final.count()
print('=' * 60)
print('RESUMEN — PIPELINE COMPLETO ACTIVIDAD 11')
print('=' * 60)
print()
print('ENTRADAS:')
print(f'  ENOE bronce (85 ZIPs) : {filas_enoe:,} filas')
print(f'  INPC mensual          : serie general, promediada por trimestre')
print(f'  IGAE mensual          : Total, promediado por trimestre')
print()
print('TRANSFORMACIONES:')
print('  1. ENOE: cast de tipos, dedup, filtro eda>=15, -1->NULL en ingresos')
print('  2. INPC: filtro subíndice general, promedio trimestral')
print('  3. IGAE: filtro Total, promedio trimestral')
print('  4. Join ENOE x INPC: left join por anio+trimestre')
print('  5. Join panel x IGAE: left join por anio+trimestre')
print('  6. Deflactación: salario_real e ing7c_real (base T4-2018)')
print()
print('SALIDA:')
print(f'  Panel analítico       : {filas_panel:,} filas')
print(f'  Columnas              : {len(df_final.columns)}')
rango = df_final.select(F.min('anio'), F.max('anio')).collect()[0]
print(f'  Cobertura temporal    : {rango[0]} – {rango[1]}')
print(f'  Granularidad          : persona x trimestre')
print(f'  Ubicación             : data/silver/panel_enoe_inpc/')
print('=' * 60)
print('Pipeline completado — datos listos para análisis.')
spark.stop()